In [0]:
from pyspark.sql import functions as F

members_clean = spark.table(
    "workspace.kkbox.members_clean"
)

customer_scope = spark.table(
    "workspace.kkbox.customer_scope"
)

customer_cohorts = (
    customer_scope
    .join(
        members_clean.select(
            "msno",
            "registration_date"
        ),
        on="msno",
        how="left"
    )
    .withColumn(
        "signup_cohort",
        F.date_format(
            F.to_date("registration_date"),
            "yyyy-MM"
        )
    )
)

display(customer_cohorts.limit(10))

In [0]:
cohort_validation = (
    customer_cohorts
    .groupBy("signup_cohort")
    .agg(
        F.count("*").alias("customer_count"),
        F.sum(
            F.when(F.col("is_churn") == 1, 1).otherwise(0)
        ).alias("churned_customers")
    )
    .orderBy("signup_cohort")
)

display(cohort_validation)

In [0]:
cohort_summary = (
    customer_cohorts
    .groupBy("signup_cohort")
    .agg(
        F.count("*").alias("customer_count"),
        F.sum(
            F.when(F.col("is_churn") == 1, 1).otherwise(0)
        ).alias("churned_customers")
    )
    .withColumn(
        "churn_rate",
        F.round(
            F.col("churned_customers") / F.col("customer_count") * 100,
            2
        )
    )
    .orderBy("signup_cohort")
)

display(cohort_summary)

In [0]:
transactions_clean_v1 = spark.table(
    "workspace.kkbox.transactions_clean_v1"
)

transaction_activity = (
    transactions_clean_v1
    .select(
        "msno",
        F.date_trunc(
            "month",
            F.col("transaction_date_clean")
        ).alias("activity_month")
    )
    .dropDuplicates(["msno", "activity_month"])
)

customer_retention_base = (
    customer_cohorts
    .select(
        "msno",
        "signup_cohort",
        "is_churn"
    )
    .join(
        transaction_activity,
        on="msno",
        how="inner"
    )
)

display(customer_retention_base.limit(10))

In [0]:
cohort_retention = (
    customer_retention_base
    .filter(F.col("signup_cohort").isNotNull())
    .groupBy(
        "signup_cohort",
        "activity_month"
    )
    .agg(
        F.countDistinct("msno").alias("active_customers")
    )
    .orderBy(
        "signup_cohort",
        "activity_month"
    )
)

display(cohort_retention.limit(100))

In [0]:
cohort_activity_check = (
    customer_retention_base
    .filter(F.col("signup_cohort").isNotNull())
    .withColumn(
        "cohort_month",
        F.to_date(
            F.concat(
                F.col("signup_cohort"),
                F.lit("-01")
            )
        )
    )
    .withColumn(
        "months_since_signup",
        F.floor(
            F.months_between(
                F.col("activity_month"),
                F.col("cohort_month")
            )
        )
    )
)

display(
    cohort_activity_check
    .select(
        "signup_cohort",
        "activity_month",
        "months_since_signup"
    )
    .limit(100)
)

In [0]:
retention_counts = (
    cohort_activity_check
    .filter(
        (F.col("months_since_signup") >= 0) &
        (F.col("signup_cohort").isNotNull())
    )
    .groupBy(
        "signup_cohort",
        "months_since_signup"
    )
    .agg(
        F.countDistinct("msno").alias("active_customers")
    )
    .orderBy(
        "signup_cohort",
        "months_since_signup"
    )
)

display(retention_counts.limit(100))

In [0]:
cohort_observation_check = (
    retention_counts
    .groupBy("signup_cohort")
    .agg(
        F.min("months_since_signup").alias("first_observed_month"),
        F.max("months_since_signup").alias("last_observed_month"),
        F.countDistinct("months_since_signup").alias("observed_months")
    )
    .orderBy("signup_cohort")
)

display(cohort_observation_check)

In [0]:
cohort_baseline = (
    retention_counts
    .filter(F.col("months_since_signup") == 0)
    .select(
        "signup_cohort",
        F.col("active_customers").alias("month_0_customers")
    )
)

cohort_retention = (
    retention_counts
    .join(
        cohort_baseline,
        on="signup_cohort",
        how="inner"
    )
    .withColumn(
        "retention_rate",
        F.round(
            F.col("active_customers") /
            F.col("month_0_customers") * 100,
            2
        )
    )
    .orderBy(
        "signup_cohort",
        "months_since_signup"
    )
)

display(cohort_retention.limit(100))

In [0]:
cohort_sizes = (
    customer_cohorts
    .filter(F.col("signup_cohort").isNotNull())
    .groupBy("signup_cohort")
    .agg(
        F.countDistinct("msno").alias("cohort_customers")
    )
)

cohort_activity_rate = (
    retention_counts
    .join(
        cohort_sizes,
        on="signup_cohort",
        how="inner"
    )
    .withColumn(
        "activity_rate",
        F.round(
            F.col("active_customers") /
            F.col("cohort_customers") * 100,
            2
        )
    )
    .orderBy(
        "signup_cohort",
        "months_since_signup"
    )
)

display(cohort_activity_rate.limit(100))

In [0]:
recent_cohort_activity = (
    cohort_activity_rate
    .filter(
        (F.col("signup_cohort") >= "2015-01") &
        (F.col("months_since_signup") <= 12)
    )
    .orderBy(
        "signup_cohort",
        "months_since_signup"
    )
)

display(recent_cohort_activity)

In [0]:
cohort_churn_summary = (
    customer_cohorts
    .filter(F.col("signup_cohort").isNotNull())
    .groupBy("signup_cohort")
    .agg(
        F.countDistinct("msno").alias("cohort_customers"),
        F.sum(
            F.when(F.col("is_churn") == 1, 1).otherwise(0)
        ).alias("churned_customers")
    )
    .withColumn(
        "churn_rate",
        F.round(
            F.col("churned_customers") /
            F.col("cohort_customers") * 100,
            2
        )
    )
    .filter(F.col("signup_cohort") >= "2015-01")
    .orderBy("signup_cohort")
)

display(cohort_churn_summary)

In [0]:
month_12_activity = (
    cohort_activity_rate
    .filter(
        (F.col("months_since_signup") == 12) &
        (F.col("signup_cohort") >= "2015-01")
    )
    .select(
        "signup_cohort",
        F.col("activity_rate").alias("month_12_activity_rate")
    )
)

cohort_business_summary = (
    cohort_churn_summary
    .join(
        month_12_activity,
        on="signup_cohort",
        how="left"
    )
    .orderBy("signup_cohort")
)

display(cohort_business_summary)

In [0]:
cohort_correlation = (
    cohort_business_summary
    .filter(
        F.col("month_12_activity_rate").isNotNull()
    )
    .agg(
        F.round(
            F.corr(
                "month_12_activity_rate",
                "churn_rate"
            ),
            3
        ).alias("activity_churn_correlation")
    )
)

display(cohort_correlation)

In [0]:
cohort_period_comparison = (
    cohort_business_summary
    .filter(
        (F.col("signup_cohort") >= "2015-01") &
        (F.col("signup_cohort") <= "2015-12") &
        F.col("month_12_activity_rate").isNotNull()
    )
    .withColumn(
        "cohort_period",
        F.when(
            F.col("signup_cohort") <= "2015-06",
            "2015-01 to 2015-06"
        ).otherwise("2015-07 to 2015-12")
    )
    .groupBy("cohort_period")
    .agg(
        F.round(F.avg("churn_rate"), 2).alias("avg_churn_rate"),
        F.round(F.avg("month_12_activity_rate"), 2).alias("avg_month_12_activity_rate"),
        F.count("*").alias("cohort_count")
    )
    .orderBy("cohort_period")
)

display(cohort_period_comparison)